# Lesson 35: Agentic AI — agents vs traditional AI, architectures and memory 🤖🔁
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Define an **AI agent** and compare **agentic AI** with **traditional AI**
2. Build an agent **from scratch** with native **tool calling**
3. Compare four **agent architectures**: ReAct loop, plan-and-execute, reflection, multi-agent
4. Give an agent **short-term** and **long-term memory**

**⏱️ Timing:** what is an agent 10' | tool-calling agent 15' | architectures 20' | memory 15'

> ⚙️ **GPU required:** Runtime → Change runtime type → **T4 GPU**. Agents need a smarter model, so this lesson uses `qwen2.5:7b` (about 5 GB, 2–3 minutes to download). If that's too slow, change it to `qwen2.5:3b` — it works, but makes more tool-calling mistakes.

## 🧭 What is an agent?
A chatbot **answers**. An **agent** **gets things done**: it is given a **goal**, then repeatedly **thinks → acts (uses tools) → observes the result**, until the goal is reached.

> Analogy: a calculator vs a personal assistant. You tell the assistant "organise the class trip" — it checks the calendar, books the bus, emails parents and comes back when it's done (or stuck).

### Traditional AI vs agentic AI
| | Traditional AI | Agentic AI |
|---|---|---|
| Example | Spam filter, face unlock, one chatbot reply | "Research 3 museums, compare prices, draft an email to parents" |
| Steps | One input → one output | Many steps, chosen by the AI |
| Tools | None | Search, calculators, databases, email, code… |
| Who decides what's next? | The programmer | The **agent** (within limits) |
| Main risks | Wrong predictions | Wrong **actions**, runaway loops, cost, security |

### The parts of an agent
**LLM "brain"** + **tools** + **memory** + **a loop** + **limits / guardrails** (Lesson 29)

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:7b"     # smarter at using tools. "qwen2.5:3b" downloads faster but makes more mistakes
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install openai sentence-transformers
import json, re, warnings; warnings.filterwarnings("ignore")
from openai import OpenAI
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def llm(messages, tools=None, temperature=0):
    kwargs = {"tools": tools} if tools else {}
    return client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, **kwargs).choices[0].message

## Part 1: An agent from scratch with tool calling 🛠️
In Lesson 29 we parsed `Action: tool[...]` text ourselves. Modern models support **native tool calling**: we describe tools as JSON, and the model replies with a structured request to call one.

In [ ]:
# Our tools: plain Python functions
CLASS = {"Aroha": [78, 85, 91], "Ben": [62, 70, 58], "Chen": [88, 94, 90], "Dina": [55, 61, 72]}
def get_scores(student: str) -> str:
    return json.dumps(CLASS.get(student.title(), "unknown student"))
def list_students() -> str:
    return json.dumps(list(CLASS))
def calculate(expression: str) -> str:
    if not re.fullmatch(r"[\d\s+\-*/().,]+", expression): return "error: maths only"
    return str(round(eval(expression), 2))

TOOLS = {"get_scores": get_scores, "list_students": list_students, "calculate": calculate}

# Descriptions the model reads (JSON schema)
TOOL_SPECS = [
    {"type": "function", "function": {"name": "list_students", "description": "List all students in the class.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "get_scores", "description": "Get a student's three test scores.",
        "parameters": {"type": "object", "properties": {"student": {"type": "string"}}, "required": ["student"]}}},
    {"type": "function", "function": {"name": "calculate", "description": "Evaluate a maths expression, e.g. (78+85+91)/3.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
]

In [ ]:
def run_agent(goal, system="You are a helpful teacher's assistant. Use tools to get facts; never guess scores.", max_steps=8):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": goal}]
    for step in range(max_steps):                                  # ⚠️ always limit the loop
        reply = llm(messages, tools=TOOL_SPECS)
        messages.append(reply.model_dump(exclude_none=True))
        if not reply.tool_calls:                                   # no more tools needed → done
            return reply.content
        for call in reply.tool_calls:                              # ACT
            args = json.loads(call.function.arguments or "{}")
            try:                                                   # a bad tool call must not crash the agent:
                result = TOOLS[call.function.name](**args)         # send the error back so the model can fix it
            except Exception as e:
                result = f"error: {e}"
            print(f"  🔧 {call.function.name}({args}) → {result}")   # OBSERVE
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return "Stopped: too many steps."

print("✅", run_agent("What is Ben's average score?"))

In [ ]:
print("✅", run_agent("Which student has the highest average? Check every student."))

### 🧩 Exercise 1
1. Read the 🔧 lines. Did the agent choose sensible tools in a sensible order? Did it ever make a mistake (wrong tool name or argument) and then **recover** after seeing the error?
2. Add a tool `add_score(student, score)` that appends a new score. Ask the agent to "add 80 for Dina and tell me her new average".
3. **Safety:** which tools would you *never* give a school agent without a human approving each use? (Sending emails? Changing grades? Deleting files?)

## Part 2: Agent architectures 🏛️
| Architecture | How it works | Good for |
|---|---|---|
| **ReAct loop** (Part 1) | Think → act → observe, repeat | Short tasks with tools |
| **Plan-and-execute** | Make a full plan first, then do each step | Longer tasks; easier to check |
| **Reflection** | Draft → critique yourself → improve | Writing, code, quality |
| **Multi-agent** | Several specialised agents, often with a supervisor | Big tasks with different skills (Lessons 37–39) |

In [ ]:
# PLAN-AND-EXECUTE
def plan(goal):
    out = llm([{"role": "user", "content": f"Goal: {goal}\nWrite a numbered plan of 2–4 short steps that use these abilities: "
                 "list students, get a student's scores, calculate. Reply with the numbered steps only."}]).content
    return [re.sub(r"^\s*\d+[.)]\s*", "", l).strip() for l in out.split("\n") if re.match(r"^\s*\d+[.)]", l)]

goal = "Find every student whose average is below 65 so the teacher can offer extra help."
steps = plan(goal)
print("📋 PLAN:"); [print("  ", i + 1, s) for i, s in enumerate(steps)]
notes = ""
for i, s in enumerate(steps, 1):
    print(f"\n▶️ Step {i}: {s}")
    notes += f"\nStep {i} result: " + run_agent(f"Overall goal: {goal}\nNotes so far: {notes}\nDo only this step: {s}")
print("\n📝 FINAL:", llm([{"role": "user", "content": f"Goal: {goal}\nNotes: {notes}\nGive the final answer in one sentence."}]).content)

In [ ]:
# REFLECTION
task = "Write a 3-sentence message to parents announcing a maths help club on Thursdays at lunchtime in room B12."
draft = llm([{"role": "user", "content": task}], temperature=0.7).content
critique = llm([{"role": "user", "content": f"Task: {task}\nDraft: {draft}\nList 2 specific problems with this draft "
                 "(missing details, tone, length). Be brief."}]).content
final = llm([{"role": "user", "content": f"Task: {task}\nDraft: {draft}\nCritique: {critique}\nRewrite the draft fixing the problems."}]).content
print("📝 DRAFT:\n", draft, "\n\n🔍 CRITIQUE:\n", critique, "\n\n✨ FINAL:\n", final)

In [ ]:
# MULTI-AGENT (supervisor + specialists)
SPECIALISTS = {
    "maths_tutor": "You are a maths tutor. Explain the method step by step, briefly.",
    "writing_coach": "You are an English writing coach. Give 2 concrete tips, briefly.",
    "wellbeing": "You are a caring school counsellor. Be warm and suggest talking to a trusted adult if needed.",
}
def supervisor(message):
    choice = llm([{"role": "user", "content": f"Which specialist should handle this message: {', '.join(SPECIALISTS)}?\n"
                   f"Message: {message}\nReply with the specialist name only."}]).content.strip().lower()
    name = next((n for n in SPECIALISTS if n in choice), "wellbeing")
    answer = llm([{"role": "system", "content": SPECIALISTS[name]}, {"role": "user", "content": message}]).content
    return name, answer

for m in ["How do I solve 3x - 7 = 11?", "My essay intro is boring, help!", "I'm really nervous about exams next week."]:
    who, ans = supervisor(m)
    print(f"🧑 {m}\n👉 routed to {who}\n🤖 {ans[:250]}\n")

### 🧩 Exercise 2
1. For each architecture, count how many LLM calls it made. Which is most expensive? Most reliable?
2. Combine two: add a **reflection** step to the multi-agent system so the supervisor checks the specialist's answer before sending it.

## Part 3: Memory 🧠
| Memory type | Human example | Agent version |
|---|---|---|
| **Short-term / working** | What you just heard in class | The recent message list (limited by the context window) |
| **Long-term – semantic** | Facts you know ("my teacher is Ms Lee") | Facts stored in a vector database, recalled by similarity |
| **Long-term – episodic** | Memories of events ("last week's test went badly") | Logs of past conversations or tasks |
| **Procedural** | Knowing *how* to ride a bike | Instructions, skills and tools the agent has |

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
embedder = SentenceTransformer("all-MiniLM-L6-v2")

class LongTermMemory:
    def __init__(self): self.texts, self.vecs = [], []
    def remember(self, text):
        self.texts.append(text); self.vecs.append(embedder.encode(text, normalize_embeddings=True))
    def recall(self, query, k=2, min_score=0.3):
        if not self.texts: return []
        scores = np.array(self.vecs) @ embedder.encode(query, normalize_embeddings=True)
        return [self.texts[i] for i in scores.argsort()[::-1][:k] if scores[i] > min_score]

memory = LongTermMemory()
SHORT_TERM_LIMIT = 6                     # keep only the last 6 messages (working memory)
history = []

def memory_chat(user_msg):
    global history
    facts = llm([{"role": "user", "content": f"Extract any lasting personal fact from this message (preferences, goals, names). "
                  f"Reply NONE if there isn't one.\nMessage: {user_msg}"}]).content.strip()
    if "NONE" not in facts.upper():
        memory.remember(facts); print(f"   💾 saved to long-term memory: {facts}")
    recalled = memory.recall(user_msg)
    system = "You are a friendly study buddy. Be brief." + (f" Things you remember about the student: {recalled}" if recalled else "")
    history = (history + [{"role": "user", "content": user_msg}])[-SHORT_TERM_LIMIT:]
    answer = llm([{"role": "system", "content": system}] + history).content
    history.append({"role": "assistant", "content": answer})
    return answer

for msg in ["Hi! I'm Tama and I want to become a vet.", "My weakest subject is chemistry.",
            "Tell me a joke about penguins.", "What's 12 squared?", "Can you suggest what I should study this weekend?"]:
    print(f"🧑 {msg}\n🤖 {memory_chat(msg)}\n")

In [ ]:
# Short-term memory has forgotten the start of the chat… but long-term memory still has it
history = []        # simulate a brand-new conversation next week
print(memory_chat("Hi again! Do you remember what career I want?"))

### 🧩 Exercise 3
1. Should an agent remember **everything**? What should it **not** store? (Passwords, health details, other people's information…)
2. Add a `forget(text)` feature. Why is "the right to be forgotten" important for AI memory?

### 🚀 Challenge
Give the agent from Part 1 long-term memory so it remembers which students the teacher asked about last time.

## 📌 Summary
- **Agent** = LLM + tools + memory + a loop with limits, working towards a goal
- Agentic AI takes **multi-step actions**; traditional AI makes **single predictions**
- Architectures: **ReAct**, **plan-and-execute**, **reflection**, **multi-agent**
- Memory: **short-term** (recent messages) and **long-term** (semantic, episodic, procedural)